# Day 03 - Outlier Detection

Outliers are values that are very different from the rest of the data.

There are two types:
- Real outliers: A person earning $500,000/month is extreme but real. Keep them.
- Error outliers: A person with age = 0 is impossible. Remove or fix them.

Today we will:
- Find outliers using box plots
- Use the IQR method to calculate exactly what counts as an outlier
- Decide what to do with each one

## Step 1 - Import and load

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('cs-training-day2-cleaned.csv', index_col=0)

df_backup = df.copy()

In [ ]:
df.shape

## Step 2 - Box plot for each column

A box plot shows:
- The box = where most of the data is (middle 50%)
- The lines coming out = 1.5x the box size
- Dots outside the lines = outliers

In [ ]:
# Box plot for age
df['age'].plot(kind='box')
plt.title('Box plot - age')
plt.show()

In [ ]:
# Box plot for monthly income
df['MonthlyIncome'].plot(kind='box')
plt.title('Box plot - MonthlyIncome')
plt.show()

In [ ]:
# Box plot for debt ratio
df['DebtRatio'].plot(kind='box')
plt.title('Box plot - DebtRatio')
plt.show()

In [ ]:
# Box plot for utilization
df['RevolvingUtilizationOfUnsecuredLines'].plot(kind='box')
plt.title('Box plot - Credit Utilization')
plt.show()

## Step 3 - IQR method to find outlier boundaries

IQR = Q3 - Q1 (the size of the box in the box plot)

Lower boundary = Q1 - 1.5 x IQR
Upper boundary = Q3 + 1.5 x IQR

Anything outside these boundaries is an outlier

In [ ]:
# Let's do this for age step by step
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print('Q1:', Q1)
print('Q3:', Q3)
print('IQR:', IQR)
print('Lower boundary:', lower)
print('Upper boundary:', upper)

In [ ]:
# How many age values are outside these boundaries?
outliers_age = df[(df['age'] < lower) | (df['age'] > upper)]
print('Number of age outliers:', len(outliers_age))

In [ ]:
# What are the actual outlier age values?
outliers_age['age'].value_counts().head(20)

## Step 4 - Check each column for specific problems

In [ ]:
# Age - can't be less than 18 (can't legally take a loan)
print('People with age < 18:', (df['age'] < 18).sum())
print('Min age:', df['age'].min())
print('Max age:', df['age'].max())

In [ ]:
# Credit utilization - above 1.5 (150%) is extreme
print('Utilization above 1.5:', (df['RevolvingUtilizationOfUnsecuredLines'] > 1.5).sum())
print('Max utilization:', df['RevolvingUtilizationOfUnsecuredLines'].max())

In [ ]:
# Debt ratio - extremely high values are data errors
print('Debt ratio above 5000:', (df['DebtRatio'] > 5000).sum())
print('Max debt ratio:', df['DebtRatio'].max())

In [ ]:
# Late payment columns - values of 96 and 98 appear a lot, this is a known data error in this dataset
print('90-day late payments value counts (top 10):')
print(df['NumberOfTimes90DaysLate'].value_counts().head(10))

## Step 5 - Fix the outliers

For error outliers (impossible values): remove those rows
For real outliers (just extreme): cap them at a reasonable max (called Winsorization)

Winsorization = instead of deleting the row, just change the extreme value to the cap value

In [ ]:
rows_before = len(df)
print('Rows before:', rows_before)

In [ ]:
# Remove rows where age is less than 18
df = df[df['age'] >= 18]
print('Rows after removing bad ages:', len(df))

In [ ]:
# Cap utilization at 1.5 (150%) - values above this are extreme
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.5)
print('Max utilization now:', df['RevolvingUtilizationOfUnsecuredLines'].max())

In [ ]:
# Cap debt ratio at 99th percentile
cap_value = df['DebtRatio'].quantile(0.99)
df['DebtRatio'] = df['DebtRatio'].clip(upper=cap_value)
print('DebtRatio capped at:', cap_value)

In [ ]:
# Cap income at 99th percentile
cap_income = df['MonthlyIncome'].quantile(0.99)
df['MonthlyIncome'] = df['MonthlyIncome'].clip(upper=cap_income)
print('MonthlyIncome capped at:', cap_income)

In [ ]:
# Cap late payment columns at 10 (96/98 values are errors in this dataset)
df['NumberOfTimes90DaysLate'] = df['NumberOfTimes90DaysLate'].clip(upper=10)
df['NumberOfTime30-59DaysPastDueNotWorse'] = df['NumberOfTime30-59DaysPastDueNotWorse'].clip(upper=10)
df['NumberOfTime60-89DaysPastDueNotWorse'] = df['NumberOfTime60-89DaysPastDueNotWorse'].clip(upper=10)
print('Late payment columns capped at 10')

## Step 6 - Compare before and after for one column

In [ ]:
# Debt ratio before and after
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df_backup['DebtRatio'].clip(upper=5000).plot(kind='box')
plt.title('DebtRatio BEFORE')

plt.subplot(1, 2, 2)
df['DebtRatio'].plot(kind='box')
plt.title('DebtRatio AFTER')

plt.tight_layout()
plt.show()

## Step 7 - Save cleaned data

In [ ]:
print('Rows removed due to bad age:', rows_before - len(df))
print('Final shape:', df.shape)

In [ ]:
df.to_csv('cs-training-day3-cleaned.csv')
print('Saved!')

## My observations today (fill this yourself)

1. Which column had the most extreme outliers?
2. Why did I remove age < 18 rows but cap income instead of removing?
3. What is the difference between removing and capping?
4. What were the 96/98 values in the late payment columns?

---
Tomorrow - Day 04: Deep dive into the target variable